## Background: WordNet

WordNet is a lexical database of English words developed at Princeton University. Rather than organizing words alphabetically like a dictionary, WordNet groups words into sets of cognitive synonyms called **synsets**, each representing a single concept or meaning.

Words often appear in multiple synsets because they have multiple meanings. The word *bank* has 18 definitions in WordNet — a financial institution, a riverbank, a flight maneuver, and more. Each is a separate synset.

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import nltk
from nltk.corpus import wordnet as wn

wn.synsets("dog")[0].definition()

'a member of the genus Canis (probably descended from the common wolf) that has been domesticated by man since prehistoric times; occurs in many breeds'

### The Hypernym Tree

The key structure we care about is the **hypernym/hyponym hierarchy** — the same "is-a" tree covered in class.

- A **hypernym** is a more general concept — *carnivore* is a hypernym of *wolf*
- A **hyponym** is a more specific concept — *wolf* is a hyponym of *carnivore*

Every noun traces back up to `entity` at the root. The deeper two words share a common ancestor, the more semantically related they are — which is exactly what Wu-Palmer similarity captures.

### Word synset Ambiguity

Here's where it gets interesting. Because words have multiple synsets, *which synset you pick changes where a word lives in the tree entirely.*

In [ ]:
wn.synsets("gun", pos=wn.NOUN)
# gun.n.01 → a weapon
# gunman.n.02 → a person who shoots a gun

[Synset('gun.n.01'),
 Synset('artillery.n.01'),
 Synset('gunman.n.02'),
 Synset('gunman.n.01'),
 Synset('grease-gun.n.01'),
 Synset('accelerator.n.01'),
 Synset('gun.n.07')]

Under its default synset, *gun.n.01* is a weapon — an `artifact`. But under *gunman.n.02*, it refers to a person who shoots a gun — placing it under `person` in the hierarchy instead.

## The Idea: Ancestor Hunt

The goal is to build a word game that uses WordNet's hypernym tree as its core mechanic.

The player is shown **2 words** and has 5 guesses to name a **common ancestor** shared by both of them. The deeper the ancestor you name, the more points you earn. Guessing `entity` is always safe since every noun traces back to it, but it's worth almost nothing.

The first step was finding all common hypernyms between two words — every ancestor they share in the tree.


### The Catch: Synset Ambiguity

Not every pair of words has a deep common ancestor under their default sense. This is where word sense ambiguity becomes a feature rather than a problem.

By picking the right synset for each word, seemingly unrelated words can share a surprisingly deep ancestor. The puzzle generator tries all sense combinations and picks the pairing that produces the deepest shared hierarchy.

In [ ]:
def all_common_hypernyms(word1, word2):
    syn1 = wn.synsets(word1)[0]
    syn2 = wn.synsets(word2)[0]

    hyper1 = set(syn1.closure(lambda s: s.hypernyms()))
    hyper2 = set(syn2.closure(lambda s: s.hypernyms()))

    return hyper1.intersection(hyper2)

print(all_common_hypernyms("dog", "cat"))

{Synset('living_thing.n.01'), Synset('vertebrate.n.01'), Synset('entity.n.01'), Synset('object.n.01'), Synset('placental.n.01'), Synset('carnivore.n.01'), Synset('whole.n.02'), Synset('animal.n.01'), Synset('mammal.n.01'), Synset('chordate.n.01'), Synset('physical_entity.n.01'), Synset('organism.n.01')}


This gives us the full set of shared ancestors, but as an unordered set it's hard to reason about. What we really want is to see the hierarchy — ordered from most general to most specific. That's what how the game will work. the player is trying to find the *deepest* one.

In [ ]:
def shared_hierarchy(word1, word2):
    syn1 = wn.synsets(word1)[0]  # take the first (most common) sense of word1
    syn2 = wn.synsets(word2)[0]  # take the first (most common) sense of word2

    hyper1 = set(syn1.closure(lambda s: s.hypernyms()))
    hyper2 = set(syn2.closure(lambda s: s.hypernyms()))

    shared = hyper1.intersection(hyper2)

    # sort: most broad (small depth) → most specific (large depth)
    sorted_shared = sorted(shared, key=lambda s: s.max_depth())

    return sorted_shared

def print_shared_hierarchy(word1, word2):
    shared = shared_hierarchy(word1, word2)
    print(f"Shared hierarchy for '{word1}' and '{word2}':\n")
    for i, syn in enumerate(shared):
        indent = "  " * i
        name = syn.name().split('.')[0]
        print(f"{indent}↳ {name}")

Some word pairs share a deep, specific ancestor — making for a satisfying and challenging puzzle.

In [ ]:
print_shared_hierarchy("Adaptation", "tinting")

Shared hierarchy for 'Adaptation' and 'tinting':

↳ entity
  ↳ abstraction


Others share something more moderate — still a valid puzzle, just a lower ceiling on points.

In [ ]:
print_shared_hierarchy("scalpel", "hammer")

Shared hierarchy for 'scalpel' and 'hammer':

↳ entity
  ↳ physical_entity
    ↳ object
      ↳ whole
        ↳ artifact
          ↳ instrumentality
            ↳ device


And some pairs — under their default sense — barely share anything at all.

In [ ]:
print_shared_hierarchy("wheel", "fish")
print("----------")
print_shared_hierarchy("love", "gun")

Shared hierarchy for 'wheel' and 'fish':

↳ entity
  ↳ physical_entity
    ↳ object
      ↳ whole
----------
Shared hierarchy for 'love' and 'gun':

↳ entity


### The Catch: synset Ambiguity

Pairs like *love* and *gun* look like dead ends,thats only because we're using their default sense. Since every word has multiple synsets, the idea is to try and find all synset combinations and keeps whichever pairing produces the deepest shared hierarchy.

In [ ]:
def best_shared_hierarchy(word1, word2):
    best = []

    for syn1 in wn.synsets(word1):
        for syn2 in wn.synsets(word2):
            hyper1 = set(syn1.closure(lambda s: s.hypernyms()))
            hyper2 = set(syn2.closure(lambda s: s.hypernyms()))

            shared = hyper1.intersection(hyper2)
            sorted_shared = sorted(shared, key=lambda s: s.max_depth())

            if len(sorted_shared) > len(best):
                best = sorted_shared

    return best

In [ ]:
shared = best_shared_hierarchy("annoying", "bloodbath")

for i, syn in enumerate(shared):
    indent = "  " * i
    name = syn.name().split('.')[0]
    print(f"{indent}↳ {name}")

↳ entity
  ↳ abstraction
    ↳ psychological_feature
      ↳ event
        ↳ act


much deeper! But we also want to know *which* synsets were chosen, so we can reveal them as a hint after their first guess.

In [ ]:
def best_shared_hierarchy(word1, word2):
    best = []
    best_pair = (None, None)

    synsets1 = wn.synsets(word1, pos=wn.NOUN)
    synsets2 = wn.synsets(word2, pos=wn.NOUN)

    for syn1 in synsets1:
        for syn2 in synsets2:
            hyper1 = set(syn1.closure(lambda s: s.hypernyms()))
            hyper2 = set(syn2.closure(lambda s: s.hypernyms()))

            shared = hyper1.intersection(hyper2)
            sorted_shared = sorted(shared, key=lambda s: s.max_depth())

            if len(sorted_shared) > len(best):
                best = sorted_shared
                best_pair = (syn1, syn2)

    return best, best_pair

In [ ]:
shared, (syn1, syn2) = best_shared_hierarchy("trudge", "project")

print("Chosen meanings:\n")
print("love →", syn1.name(), "|", syn1.definition())
print("gun  →", syn2.name(), "|", syn2.definition())

print("\nShared hierarchy:\n")

for i, syn in enumerate(shared):
    indent = "  " * i
    name = syn.name().split('.')[0]
    print(f"{indent}↳ {name}")

Chosen meanings:

love → trudge.n.01 | a long difficult walk
gun  → undertaking.n.01 | any piece of work that is undertaken or attempted

Shared hierarchy:

↳ entity
  ↳ abstraction
    ↳ psychological_feature
      ↳ event
        ↳ act


*love* and *gun* look like they have nothing in common — but under the right reading, they're both a type of `person`. This is the core trick i want to leverage. the player has to figure out not just the ancestor, but which sense of each word the puzzle intends.



# Synset Hypernymy

Synset Hypernymy is a word game built on top of WordNet's hypernym tree. The player is shown two words and has 7 guesses to name as many of their common ancestors as possible. The deeper the ancestor, the more points it's worth.

### Scoring

Every correct ancestor adds to your score. Points are awarded based on how deep the ancestor sits in WordNet's hierarchy — `entity` is always valid but barely worth anything, while a specific ancestor like `carnivore` or `person` earns full points.

```python
def points_for(depth, max_depth):
    return round((depth / max_depth) * 100)

# entity   → depth 1  →  ~8 pts
# organism → depth 5  → ~55 pts
# person   → depth 7  → 100 pts
```

### Hint System

The two words shown to the player are not always what they seem — WordNet assigns multiple meanings to most words, and the puzzle uses whichever sense produces the deepest shared hierarchy. After the first guess, the intended noun sense for each word is revealed. This reframes the puzzle and gives the player a better shot at finding the deeper ancestors.

The game ends when the player either finds all common ancestors or runs out of guesses.

### Getting Two Words

The foundation of the game is finding two words that share a deep common ancestor in WordNet. We try all possible synset combinations for both words and keep the pairing that produces the longest shared hierarchy.

This also returns the specific synset pair that produced the best result — we'll need those definitions later for the hint system.

In [ ]:
def best_shared_hierarchy(word1, word2):
    best = []
    best_pair = (None, None)

    synsets1 = wn.synsets(word1, pos=wn.NOUN)
    synsets2 = wn.synsets(word2, pos=wn.NOUN)

    for syn1 in synsets1:
        for syn2 in synsets2:
            hyper1 = set(syn1.closure(lambda s: s.hypernyms()))
            hyper2 = set(syn2.closure(lambda s: s.hypernyms()))

            shared = hyper1.intersection(hyper2)
            sorted_shared = sorted(shared, key=lambda s: s.max_depth())

            if len(sorted_shared) > len(best):
                best = sorted_shared
                best_pair = (syn1, syn2)

    return best, best_pair

### Puzzle Generation

Not every pair of random words makes for a good puzzle. We need at least 5 common ancestors to give the player meaningful choices. We keep pulling random nouns from WordNet until we find a pair that clears that bar.


`max_attempts` keeps it from running forever if it gets unlucky. Once we have a valid pair we store the hint dictionary alongside the words and ancestor.

In [ ]:
import random

def get_random_noun():
    nouns = list(wn.all_synsets(pos=wn.NOUN))
    synset = random.choice(nouns)
    return synset.lemmas()[0].name().replace('_', ' ')

def generate_puzzle(min_ancestors=5, max_attempts=100):
    for _ in range(max_attempts):
        word1 = get_random_noun()
        word2 = get_random_noun()

        shared, pair = best_shared_hierarchy(word1, word2)

        if len(shared) >= min_ancestors:
            syn1, syn2 = pair
            hint = {
                word1: {"synset": syn1.name(), "definition": syn1.definition()},
                word2: {"synset": syn2.name(), "definition": syn2.definition()}
            }
            return word1, word2, shared, hint

    return None

### Hint System

The hint is already baked into the puzzle — `generate_puzzle` stores the intended synset definition for each word when it finds a valid pair. We just hold it back until after the first guess.

In [ ]:
def reveal_hint(hint):
    print("Hint — intended meanings:\n")
    for word, info in hint.items():
        print(f"{word} → {info['synset']} | {info['definition']}")

### Scoring

Points are awarded based on how deep the ancestor sits in WordNet's hierarchy. The deepest ancestor in the shared chain sets the ceiling — finding it earns 100 points. Shallower ancestors earn proportionally less. Every correct guess accumulates to the final score.

In [ ]:
def points_for(depth, max_depth):
    return round(((depth + 1) / (max_depth + 1)) * 100)

### Game Loop

Now we put it all together. The player guesses ancestors one at a time, accumulating points for each correct one. The hint reveals after the first guess regardless of whether it was right or wrong. The game ends when the player finds all ancestors or runs out of guesses.

In [ ]:
def play_round():
    puzzle = generate_puzzle()

    if not puzzle:
        print("Couldn't generate a valid puzzle. Try again.")
        return

    word1, word2, shared, hint = puzzle
    ancestor_names = [s.name().split('.')[0].replace('_', ' ') for s in shared]
    max_depth = shared[-1].max_depth()

    guesses_left = len(ancestor_names) + 2
    hint_revealed = False
    score = 0
    found = []

    print(f"\n{word1.upper()} and {word2.upper()}\n")
    print(f"There are {len(ancestor_names)} ancestors to find. You have {guesses_left} guesses.\n")

    while guesses_left > 0 and len(found) < len(ancestor_names):
        guess = input(f"Guess ({guesses_left} left | score: {score} pts): ").strip().lower()

        if guess in ancestor_names and guess not in found:
            depth = shared[ancestor_names.index(guess)].max_depth()
            pts = points_for(depth, max_depth)
            score += pts
            found.append(guess)
            print(f"✓ {guess} | depth {depth} | +{pts} pts")

            if len(found) == len(ancestor_names):
                print(f"\nYou found them all! Final score: {score} pts")
                print(f"\nFull ancestor chain:\n")
                for i, syn in enumerate(shared):
                    name = syn.name().split('.')[0].replace('_', ' ')
                    indent = "  " * i
                    print(f"{indent}✓ {name} | depth {syn.max_depth()}")
                break

        elif guess in found:
            print("Already found that one.")

        else:
            guesses_left -= 1
            print(f"Not a common ancestor. {guesses_left} guesses left.")

        if not hint_revealed:
            hint_revealed = True
            print()
            reveal_hint(hint)
            print()

    if guesses_left == 0 and len(found) < len(ancestor_names):
        print(f"\nOut of guesses! Final score: {score} pts")
        print(f"\nFull ancestor chain:\n")
        for i, syn in enumerate(shared):
            name = syn.name().split('.')[0].replace('_', ' ')
            indent = "  " * i
            found_marker = "✓" if name in found else "✗"
            print(f"{indent}{found_marker} {name} | depth {syn.max_depth()}")

play_round()


GENT and JACKSMELT

There are 6 ancestors to find. You have 8 guesses.

Guess (8 left | score: 0 pts): person
Not a common ancestor. 7 guesses left.

Hint — intended meanings:

gent → gent.n.01 | informal abbreviation of `gentleman'
jacksmelt → jacksmelt.n.01 | a relatively large silversides of the Pacific coast of North America (known to reach 18 inches in length)

Guess (7 left | score: 0 pts): Person
Not a common ancestor. 6 guesses left.
Guess (6 left | score: 0 pts): act
Not a common ancestor. 5 guesses left.
Guess (5 left | score: 0 pts): land
Not a common ancestor. 4 guesses left.
Guess (4 left | score: 0 pts): object
✓ object | depth 2 | +50 pts
Guess (4 left | score: 50 pts): entiy
Not a common ancestor. 3 guesses left.
Guess (3 left | score: 50 pts): entity
✓ entity | depth 0 | +17 pts
Guess (3 left | score: 67 pts): living being
Not a common ancestor. 2 guesses left.
Guess (2 left | score: 67 pts): item
Not a common ancestor. 1 guesses left.
Guess (1 left | score: 67 pts): 

Sagar, jacob, Alex